### Colab setup

In [1]:
%pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.7 MB/s eta 0:00:00


In [3]:
%mkdir -p /content/data/raw

!curl -L -o /content/data/raw/traffic-vehicles-object-detection.zip https://www.kaggle.com/api/v1/datasets/download/saumyapatel/traffic-vehicles-object-detection

!unzip -qo /content/data/raw/*.zip -d /content/data/raw/traffic-vehicles-object-detection
%rm /content/data/raw/*.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  606M  100  606M    0     0  20.3M      0  0:00:29  0:00:29 --:--:-- 21.9M


### Idea
- this will use YoLov8s and try to identify all the class in the predefind dataset given [Here](../data/raw/traffic-vehicles-object-detection/Traffic%20Dataset/)
- it would return the count of vehical in each image

In [15]:
import os
import sys
import json
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import cv2
import yaml
from PIL import Image

# YOLO imports
from ultralytics import YOLO
import torch
# Reproducibility
random.seed(42)
np.random.seed(42)


## Step 1: Dataset Configuration
Set paths and select 500 images for efficient fine-tuning

In [16]:
# Dataset paths
DATASET_ROOT = Path("/content/data/raw/traffic-vehicles-object-detection/Traffic Dataset")
TRAIN_IMAGES = DATASET_ROOT / "images" / "train"
TRAIN_LABELS = DATASET_ROOT / "labels" / "train"
# Output paths for selected subset
SUBSET_DIR = DATASET_ROOT
TRAIN_OUT_IMAGES = DATASET_ROOT / "images" / "train"
TRAIN_OUT_LABELS = DATASET_ROOT / "labels" / "train"
VAL_OUT_IMAGES   = DATASET_ROOT / "images" / "val"
VAL_OUT_LABELS   = DATASET_ROOT / "labels" / "val"




In [17]:
labels_dirs = [
    DATASET_ROOT / "labels" / "train",
    DATASET_ROOT / "labels" / "val",
]

# vehicle classes to keep
KEEP_IDS = {0, 3, 4, 5, 6}

class_ids = set()
removed_boxes = 0
kept_boxes = 0
bad_lines = 0

for labels_dir in labels_dirs:
    for lf in labels_dir.glob("*.txt"):
        new_lines = []

        for line in lf.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()
            if len(parts) < 5:
                bad_lines += 1
                continue

            try:
                cid = int(float(parts[0]))
            except Exception:
                bad_lines += 1
                continue

            # remove number plate classes
            if cid not in KEEP_IDS:
                removed_boxes += 1
                continue

            # map all vehicle classes -> 0
            parts[0] = "0"
            new_lines.append(" ".join(parts))
            kept_boxes += 1
            class_ids.add(0)

        lf.write_text("\n".join(new_lines))

print("Remaining class IDs:", sorted(class_ids))  # should be [0]
print("Kept vehicle boxes:", kept_boxes)
print("Removed plate boxes:", removed_boxes)
print("Bad lines:", bad_lines)

Remaining class IDs: [0]
Kept vehicle boxes: 9171
Removed plate boxes: 0
Bad lines: 0


### Step 3: Create YOLO Data Configuration

In [18]:
# Only Detection Count of the vehical
NAMES = ["vehicle"]
assert len(NAMES) == 1, "NAMES must have exactly 7 class names (id 0..6)."

# data.yaml Creation
data_yaml = {
    "path": str(SUBSET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": NAMES,
}

yaml_path = SUBSET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"\n✓ YAML config created: {yaml_path}")
print(yaml.safe_dump(data_yaml, sort_keys=False))


✓ YAML config created: /content/data/raw/traffic-vehicles-object-detection/Traffic Dataset/data.yaml
path: /content/data/raw/traffic-vehicles-object-detection/Traffic Dataset
train: images/train
val: images/val
nc: 1
names:
- vehicle



### Step 4: Train YOLOv8 Model (Minimum Resources)

In [19]:
from functools import cache
# Model config
MODEL_NAME = "yolov8s"  # nano model - minimum resources
EPOCHS = 80 # reduced epochs for efficiency
IMG_SIZE = 736  # smaller image size for speed
BATCH_SIZE = 16  # small batch size for memory efficiency

# Load pretrained model (samll - minimal resources)
model = YOLO(f'{MODEL_NAME}.pt')

# Train with resource optimization
results = model.train(
    plots=True,
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device= 0,  # Auto-select device
    patience=15,  # Early stopping
    lr0=0.001,          # slightly lower LR to avoid overfitting spikes
    cos_lr=True,        # smooth LR decay
    save=True,

    workers=4,
    cache=True,
    optimizer='AdamW',
    freeze=10,
    close_mosaic=10,
      # Augmentations tuned for traffic scenes
    hsv_h=0.01,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=3.0,        # vehicles rarely rotate
    translate=0.07,
    scale=0.4,
    shear=0.0,
    flipud=0.0,         # never flip traffic vertically
    fliplr=0.5,         # horizontal flip is realistic
    weight_decay=0.0005,
    dropout=0.0,        # YOLO head usually fine without dropout
)

print("\n✓ Training completed")
print(f"Results saved to: {results.save_dir if hasattr(results, 'save_dir') else 'runs/detect'}")

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data/raw/traffic-vehicles-object-detection/Traffic Dataset/data.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.01, hsv_s=0.6, hsv_v=0.4, imgsz=736, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, o

### Step 5: Evaluation & Metrics

In [22]:
# Step 5: Evaluation & Metrics (robust to per-class arrays)

def to_list(x):
    """Convert tensors/np arrays/lists to a plain Python list, else return None."""
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    if isinstance(x, np.ndarray):
        return x.astype(float).tolist()
    if isinstance(x, (list, tuple)):
        return [float(v) for v in x]
    return None

def to_scalar(x, reduce="mean"):
    """
    Convert value to a float scalar.
    If x is array-like, reduce it (mean by default).
    """
    if x is None:
        return None

    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()

    if isinstance(x, np.ndarray):
        if x.size == 1:
            return float(x.reshape(-1)[0])
        if reduce == "mean":
            return float(np.nanmean(x))
        if reduce == "max":
            return float(np.nanmax(x))
        if reduce == "min":
            return float(np.nanmin(x))
        raise ValueError(f"Unknown reduce={reduce}")

    if isinstance(x, (list, tuple)):
        x = np.array(x, dtype=float)
        if x.size == 1:
            return float(x.reshape(-1)[0])
        return float(np.nanmean(x)) if reduce == "mean" else float(np.nanmax(x))

    # plain python number
    try:
        return float(x)
    except Exception:
        return None

def evaluate_model(model: YOLO, yaml_path: Path) -> dict:
    metrics = model.val(data=str(yaml_path), split="val", verbose=False)

    box = getattr(metrics, "box", None)

    p = getattr(box, "p", None) if box is not None else getattr(metrics, "p", None)
    r = getattr(box, "r", None) if box is not None else getattr(metrics, "r", None)
    map50 = getattr(box, "map50", None) if box is not None else getattr(metrics, "map50", None)
    map5095 = getattr(box, "map", None) if box is not None else getattr(metrics, "map", None)
    per_class_map5095 = getattr(box, "maps", None) if box is not None else None

    precision = to_scalar(p)
    recall = to_scalar(r)

    # F1 score is useful for counting quality
    f1 = None
    if precision is not None and recall is not None and (precision + recall) > 0:
        f1 = 2 * (precision * recall) / (precision + recall)

    results = {
        "precision": precision,
        "recall": recall,
        "F1_score": f1,
        "mAP50": to_scalar(map50),
        "mAP50-95": to_scalar(map5095),

        # debugging info (optional but useful)
        "precision_per_class": to_list(p),
        "recall_per_class": to_list(r),
        "mAP50-95_per_class": to_list(per_class_map5095),
    }

    return results
# ---- Run evaluation ----
evaluation = evaluate_model(model, f"{DATASET_ROOT}/data.yaml")

print("\n" + "=" * 70)
print("EVALUATION RESULTS (val split)")
print("=" * 70)
for k, v in evaluation.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")
    else:
        print(f"{k:25s}: {v}")
print("=" * 70)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2310.7±604.0 MB/s, size: 248.2 KB)
val: Scanning /content/data/raw/traffic-vehicles-object-detection/Traffic Dataset/labels/val.cache... 185 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185 55.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.1it/s 5.6s
                   all        185       1645      0.915      0.889      0.945      0.734
Speed: 4.3ms preprocess, 8.7ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to /content/runs/detect/val5

EVALUATION RESULTS (val split)
precision                : 0.9149
recall                   : 0.8885
F1_score                 : 0.9015
mAP50                    : 0.9453
mAP50-95                 : 0.7338
precision_per_class      :

### Step 7: Test on Sample Images

In [23]:
# Test inference on validation images
VAL_IMAGES = DATASET_ROOT / "images" / "test"
test_samples = list(VAL_IMAGES.glob('*.*'))[:10]  # First 3 images

print("\nTesting inference on sample images:")
print("="*60)

for img_path in test_samples:
    # Run inference
    results = model.predict(str(img_path), conf=0.5, verbose=False)

    # Extract detections
    result = results[0]
    num_detections = len(result.boxes) if hasattr(result, 'boxes') else 0

    print(f"\n📷 {img_path.name}")
    print(f"   Detections: {num_detections} vehicles")

    if num_detections > 0:
        # Get confidence scores
        confidences = result.boxes.conf.cpu().numpy() if hasattr(result.boxes, 'conf') else []
        avg_conf = float(np.mean(confidences)) if len(confidences) > 0 else 0
        print(f"   Avg Confidence: {avg_conf:.4f}")

print("\n✓ Inference test completed")
print("="*60)

# Save model
model_save_path = Path("/content/models/traffic_detection_yolov8s.pt")
model_save_path.parent.mkdir(parents=True, exist_ok=True)
model.save(str(model_save_path))
print(f"\n✓ Model saved to: {model_save_path}")


Testing inference on sample images:

📷 00 (22).png
   Detections: 8 vehicles
   Avg Confidence: 0.8291

📷 00 (222).jpg
   Detections: 18 vehicles
   Avg Confidence: 0.8250

📷 00 (74).png
   Detections: 4 vehicles
   Avg Confidence: 0.8899

📷 00 (48).png
   Detections: 6 vehicles
   Avg Confidence: 0.8552

📷 00 (211).jpg
   Detections: 2 vehicles
   Avg Confidence: 0.8020

📷 00 (98).png
   Detections: 1 vehicles
   Avg Confidence: 0.9137

📷 00 (354).jpg
   Detections: 4 vehicles
   Avg Confidence: 0.7700

📷 00 (77).png
   Detections: 2 vehicles
   Avg Confidence: 0.9233

📷 00 (93).png
   Detections: 7 vehicles
   Avg Confidence: 0.8384

📷 00 (92).png
   Detections: 7 vehicles
   Avg Confidence: 0.8737

✓ Inference test completed

✓ Model saved to: /content/models/traffic_detection_yolov8s.pt
